# 00 — Dataset analysis

Audits all prepared Pascal-Part-116 splits.


In [ ]:
from pathlib import Path
import json
import os
import sys

from IPython.display import Image, display

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "datasets").is_dir() and (path / "final_model").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the repository or final_training_notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RUN_ID = os.environ.get("FINAL_TRAINING_RUN_ID", "manual")
FRESH_TRAINING = True
RESULT_ROOT = PROJECT_ROOT / "training_results"
POINT_ROOT = RESULT_ROOT
print("Project:", PROJECT_ROOT)
print("Run ID:", RUN_ID)


## Dataset-analysis implementation


In [ ]:
import json
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from datasets import PascalPart116Dataset

def run_id():
    return os.environ.get("FINAL_TRAINING_RUN_ID", "manual")

def results_root():
    path = PROJECT_ROOT / "training_results"
    path.mkdir(parents=True, exist_ok=True)
    return path

def run_data_analysis() -> pd.DataFrame:
    output = results_root() / "data_analysis"
    output.mkdir(parents=True, exist_ok=True)
    rows = []
    part_rows = []
    for split in ("train_seen", "validation_seen", "test_seen", "test_unseen"):
        dataset = PascalPart116Dataset(split=split)
        records = dataset.records
        rows.append(
            {
                "split": split,
                "queries": len(records),
                "images": len({record["image_id"] for record in records}),
                "object_classes": len({record["object_name"] for record in records}),
                "part_queries": len({record["query"] for record in records}),
            }
        )
        for record in records:
            part_rows.append(
                {
                    "split": split,
                    "query": record["query"],
                    "object_name": record["object_name"],
                    "part_name": record["part_name"],
                    "part_to_object_ratio": record["target_part_pixels"] / max(record["object_pixels"], 1),
                }
            )
    overview = pd.DataFrame(rows)
    detail = pd.DataFrame(part_rows)
    overview.to_csv(output / "split_overview.csv", index=False)
    detail.groupby(["split", "query"], as_index=False).agg(
        samples=("query", "size"), mean_part_ratio=("part_to_object_ratio", "mean")
    ).to_csv(output / "query_distribution.csv", index=False)
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    overview.set_index("split")["queries"].plot.bar(ax=axes[0], title="Query-level samples")
    detail.loc[detail.split == "train_seen", "query"].value_counts().head(15).sort_values().plot.barh(
        ax=axes[1], title="Most frequent training queries"
    )
    axes[0].set_ylabel("Samples")
    axes[1].set_xlabel("Samples")
    figure.tight_layout()
    figure.savefig(output / "dataset_overview.png", dpi=180, bbox_inches="tight")
    plt.close(figure)
    (output / "config.json").write_text(
        json.dumps({"run_id": run_id(), "training": False, "purpose": "pre-training dataset audit"}, indent=2) + "\n"
    )
    print(overview.to_string(index=False))
    return overview


## Run analysis and display saved graph


In [ ]:
RESULT_DIR = RESULT_ROOT / "data_analysis"
overview = run_data_analysis()
display(overview)
display(Image(filename=str(RESULT_DIR / "dataset_overview.png")))
